# 02 - PhoBERT Sentiment Pipeline

This notebook is a thin CLI runner for the local sentiment pipeline.
It does not own dataset preparation, annotation logic, classifier training, inference, or validation.

Supported flow:
- prepare a normalized article-level corpus
- export a review sample for manual or external annotation
- merge reviewed labels back into a labeled parquet
- train a 3-label classifier checkpoint
- run CafeF inference and validation


## 0 - Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

IS_KAGGLE = Path('/kaggle').exists()
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

def run_command(*args: str, cwd: Path | None = None) -> None:
    cmd = [str(arg) for arg in args]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd or ROOT, check=True)

def run_module(module: str, *args: str) -> None:
    run_command(sys.executable, '-m', module, *args)

print('IS_KAGGLE:', IS_KAGGLE)
print('ROOT     :', ROOT)


## 1 - Runtime Configuration

In [ ]:
TRAINING_INPUT = ROOT / 'data' / 'main' / 'cafef' / 'training_input.parquet'
ANNOTATION_FILE = ROOT / 'data' / 'main' / 'cafef' / 'training_annotations.csv'
TRAINING_OUTPUT = ROOT / 'data' / 'main' / 'cafef' / 'training_corpus.parquet'
MERGED_LABELED_OUTPUT = ROOT / 'data' / 'main' / 'cafef' / 'training_labeled.parquet'
ANNOTATION_SAMPLE_OUTPUT = ROOT / 'data' / 'main' / 'cafef' / 'annotation_sample.csv'
MODEL_DIR = ROOT / 'models' / 'phobert-sentiment' / 'latest'
CAFEF_ARTICLES = ROOT / 'data' / 'main' / 'processed' / 'articles_clean.parquet'
CAFEF_INPUT = ROOT / 'data' / 'main' / 'cafef' / 'cafef_input.parquet'
SENTIMENT_OUTPUT = ROOT / 'data' / 'main' / 'processed' / 'article_sentiment_scores.parquet'
DAILY_NEWS = ROOT / 'data' / 'main' / 'processed' / 'daily_news_prices.parquet'

BASE_MODEL = os.getenv('PHOBERT_BASE_MODEL', 'vinai/phobert-base-v2')
BATCH_SIZE = 16
MAX_LENGTH = 256
EPOCHS = 2
LEARNING_RATE = 2e-5
SEED = 42

for path in [TRAINING_OUTPUT.parent, MODEL_DIR.parent, SENTIMENT_OUTPUT.parent]:
    path.mkdir(parents=True, exist_ok=True)


## 2 - Optional Kaggle Install

Use this only when the notebook is running in a fresh Kaggle runtime and the repo dependencies are not installed yet.

In [ ]:
if IS_KAGGLE:
    run_command(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt')
else:
    print('Skipping pip install outside Kaggle.')


## 3 - Prepare A Normalized Training Corpus

In [ ]:
run_module(
    'src.sentiment.prepare_training_data',
    '--input-file', str(TRAINING_INPUT),
    '--output-file', str(TRAINING_OUTPUT),
)
pd.read_parquet(TRAINING_OUTPUT).head(3)


## 4 - Export Annotation Sample And Merge Reviewed Labels

The review step stays file-based. Annotate `ANNOTATION_SAMPLE_OUTPUT` outside the notebook, then run the merge cell.

In [ ]:
run_module(
    'src.sentiment.sample_annotation',
    '--input-file', str(TRAINING_OUTPUT),
    '--output-file', str(ANNOTATION_SAMPLE_OUTPUT),
    '--sample-size', '500',
    '--seed', str(SEED),
)
pd.read_csv(ANNOTATION_SAMPLE_OUTPUT).head(3)


In [ ]:
run_module(
    'src.sentiment.merge_annotations',
    '--corpus-file', str(TRAINING_OUTPUT),
    '--annotations-file', str(ANNOTATION_FILE),
    '--output-file', str(MERGED_LABELED_OUTPUT),
    '--seed', str(SEED),
)
pd.read_parquet(MERGED_LABELED_OUTPUT)[['article_id', 'label', 'split']].head(5)


## 5 - Train The Classifier

In [ ]:
run_module(
    'src.sentiment.train_classifier',
    '--labeled-input', str(MERGED_LABELED_OUTPUT),
    '--output-dir', str(MODEL_DIR),
    '--base-model', str(BASE_MODEL),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--max-length', str(MAX_LENGTH),
    '--seed', str(SEED),
)
pd.read_json(MODEL_DIR / 'evaluation.json')


## 6 - Run CafeF Inference And Validation

In [ ]:
run_module(
    'src.sentiment.run_pipeline',
    '--mode', 'infer',
    '--model-dir', str(MODEL_DIR),
    '--cafef-input', str(CAFEF_ARTICLES),
    '--cafef-prepared-output', str(CAFEF_INPUT),
    '--sentiment-output', str(SENTIMENT_OUTPUT),
    '--daily-news-file', str(DAILY_NEWS),
    '--batch-size', str(BATCH_SIZE),
    '--max-length', str(MAX_LENGTH),
)
pd.read_parquet(SENTIMENT_OUTPUT).head(5)


## 7 - One-Command Runs

In [ ]:
print('Train only:')
print(f"{sys.executable} -m src.sentiment.run_pipeline --mode train --training-input {TRAINING_INPUT} --labeled-input {ANNOTATION_FILE} --model-dir {MODEL_DIR}")
print('Infer only:')
print(f"{sys.executable} -m src.sentiment.run_pipeline --mode infer --model-dir {MODEL_DIR}")
print('Full run:')
print(f"{sys.executable} -m src.sentiment.run_pipeline --mode full --training-input {TRAINING_INPUT} --labeled-input {ANNOTATION_FILE} --model-dir {MODEL_DIR}")
